# Change a person's ACL, then delete them

`PATCH` with `native_acl` rewrites the snapshot. A person write bumps the
version, which is different from a policy ACL change, where the version stays
and only the snapshot hash moves.

The delete is visible only because the id leaves `/ids`. The reconcile sweep
removes `raw/hr/person/{id}.json`.


In [1]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [2]:
PERSON_ID = "emp-900003"
KEY = f"raw/hr/person/{PERSON_ID}.json"
PAYLOAD = {
    "person_id": PERSON_ID,
    "email": "emp-900003@example.com",
    "full_name": "Demo Person",
    "team": "people",
    "role": "People Partner",
    "region": "DE",
    "manager_id": None,
    "native_acl": ["hr_employee"],
}
hr_demo.delete_if_present("person", PERSON_ID)
created = hr_demo.admin("POST", "/admin/people", PAYLOAD)
print(created["person_id"], "version", created["version"], created["native_acl"])


emp-900003 version 1 ['hr_employee']


In [3]:
import json

def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    if not all(meta.get(name) for name in hr_demo.META_KEYS):
        return False
    if meta.get(hr_demo.META_VERSION) != str(created["version"]):
        return False
    raw = hr_demo.object_body(KEY)
    record = json.loads(raw)
    leaked = set(record) & hr_demo.FORBIDDEN
    return raw == hr_demo.canonical(record) and not leaked

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "person snapshot")
record = json.loads(hr_demo.object_body(KEY))
assert record == hr_demo.admin("GET", f"/people/{PERSON_ID}")
print(sorted(record))
print(hr_demo.snapshot(KEY)["Metadata"])


['email', 'full_name', 'manager_id', 'native_acl', 'person_id', 'region', 'role', 'team', 'updated_at', 'version']
{'kbs-content-hash': 'a427db3a5c631a1f493433dc5ee49afd64595bbbdf0abfc04d8c8a87d15af80b', 'kbs-native-acl': 'hr_employee', 'kbs-snapshot-at': '2026-10-16T16:54:57Z', 'kbs-source-updated-at': '2026-10-16T16:54:51.486066Z', 'kbs-version': '1'}


## ACL change


In [4]:
import json

before_version = hr_demo.snapshot(KEY)["Metadata"][hr_demo.META_VERSION]
changed = hr_demo.admin(
    "PATCH",
    f"/admin/people/{PERSON_ID}",
    {"native_acl": ["hr_policy_admin"]},
)
assert changed["native_acl"] == ["hr_policy_admin"]
assert changed["version"] == int(before_version) + 1

def acl_written() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    if meta.get(hr_demo.META_ACL) != "hr_policy_admin":
        return False
    if meta.get(hr_demo.META_VERSION) != str(changed["version"]):
        return False
    record = json.loads(hr_demo.object_body(KEY))
    return record["native_acl"] == ["hr_policy_admin"]

hr_demo.wait_until(acl_written, hr_demo.WRITE_WAIT, "person acl snapshot")
print(hr_demo.snapshot(KEY)["Metadata"])


{'kbs-content-hash': 'bf6e2506052fad10ca5a00478749495a94a88d2087b2cbff8caa744c59dd755a', 'kbs-native-acl': 'hr_policy_admin', 'kbs-snapshot-at': '2026-10-16T16:55:07Z', 'kbs-source-updated-at': '2026-10-16T16:54:57.777992Z', 'kbs-version': '2'}


## Delete


In [5]:
deleted_before = hr_demo.metric("kbs_snapshots_deleted_total", source_id="hr")
hr_demo.admin("DELETE", f"/admin/people/{PERSON_ID}")
missing = hr_demo.expect_status("GET", f"/people/{PERSON_ID}", 404)
print("source", missing.status_code)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "delete sweep",
)
if deleted_before is None:
    print("connector metrics not reachable; S3 versions are the check")
else:

    def counted() -> bool:
        current = hr_demo.metric("kbs_snapshots_deleted_total", source_id="hr")
        return current is not None and current > deleted_before

    hr_demo.wait_until(counted, 30, "delete metric")
    deleted_after = hr_demo.metric("kbs_snapshots_deleted_total", source_id="hr")
    print("kbs_snapshots_deleted_total", deleted_before, "->", deleted_after)


source 404
kbs_snapshots_deleted_total 2.0 -> 3.0


## Cleanup


In [6]:
hr_demo.delete_if_present("person", PERSON_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)


removed raw/hr/person/emp-900003.json
